<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/TextCleaner2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re

#INPUT_FILE  = "BR_Curated_17.txt"
INPUT_FILE  = "BR_Curated_19.txt"
OUTPUT_FILE = "BR_Sieve_A_Mojibake2.txt"

# Characters commonly left behind by broken UTF-8 / Windows encoding
suspicious_chars = [
    "Ã", "Â", "Æ", "â", "€", "™", "œ", "ž",
    "ƒ", "‚", "Å", "¤", "¢", "¬", "§", "�"
]

pattern = re.compile(
    "|".join(re.escape(x) for x in suspicious_chars)
)

hits = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        found = pattern.findall(line)

        if found:
            hits.append((line_no, found, line.rstrip()))

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    out.write("BRAHA — SIEVE A: ENCODING / MOJIBAKE\n")
    out.write("=" * 100 + "\n\n")

    for line_no, found, line in hits:
        chars = " ".join(sorted(set(found)))

        out.write(
            f"LINE {line_no}   SUSPECT: {chars}\n"
        )
        out.write(line + "\n")
        out.write("-" * 100 + "\n")

print("=" * 60)
print("BRAHA — SIEVE A: ENCODING / MOJIBAKE")
print("=" * 60)
print(f"Lines flagged : {len(hits)}")
print(f"Output file   : {OUTPUT_FILE}")
print("=" * 60)

BRAHA — SIEVE A: ENCODING / MOJIBAKE
Lines flagged : 0
Output file   : BR_Sieve_A_Mojibake2.txt


In [2]:
import re

INPUT_FILE    = "BR_Curated_17.txt"
OUTPUT_FILE   = "BR_Curated_18.txt"
RESIDUAL_FILE = "BR_Sieve_A_Residual.txt"

BAD = "Ã¢â‚¬Â¢"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read()

original_count = text.count(BAD)

# ------------------------------------------------------------
# 1. BULLETS
#    Mojibake appearing at the beginning of a line,
#    allowing leading whitespace.
# ------------------------------------------------------------

text, n_bullets = re.subn(
    rf"(?m)^(\s*){re.escape(BAD)}\s*",
    r"\1• ",
    text
)


# ------------------------------------------------------------
# 2. DEGREE SIGNS
#
# Examples:
#   5Ã¢â‚¬Â¢ Leo 23'  -> 5° Leo 23'
#   00Ã¢â‚¬Â¢-06...   -> 00°-06...
#
# Require a digit immediately before the mojibake.
# ------------------------------------------------------------

text, n_degrees = re.subn(
    rf"(?<=\d){re.escape(BAD)}",
    "°",
    text
)


# ------------------------------------------------------------
# 3. POSSESSIVE / CONTRACTION MARK
#
# Examples:
#   one's
#   person's
#
# OCR form:
#   oneÃ¢â‚¬Â¢s
#
# Only replace when BAD occurs between a letter and s.
# ------------------------------------------------------------

text, n_apostrophes = re.subn(
    rf"(?<=[A-Za-z]){re.escape(BAD)}(?=s\b)",
    "'",
    text
)


# ------------------------------------------------------------
# 4. WRITE INTERMEDIATE CLEAN FILE
#
# IMPORTANT:
# We deliberately DO NOT automatically repair BAD occurring
# elsewhere inside words:
#
#   techÃ¢â‚¬Â¢ nically
#   conÃ¢â‚¬Â¢ stantly
#
# Those go to the residual file for human inspection.
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(text)


# ------------------------------------------------------------
# 5. CREATE RESIDUAL SIEVE
# ------------------------------------------------------------

residuals = []

for line_no, line in enumerate(text.splitlines(), start=1):
    if BAD in line:
        residuals.append((line_no, line))

with open(RESIDUAL_FILE, "w", encoding="utf-8") as out:
    out.write("BRAHA — SIEVE A RESIDUAL MOJIBAKE\n")
    out.write("=" * 100 + "\n\n")

    for line_no, line in residuals:
        out.write(f"LINE {line_no}\n")
        out.write(line + "\n")
        out.write("-" * 100 + "\n")


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 65)
print("BRAHA — SIEVE A REPAIR")
print("=" * 65)

print(f"Original mojibake occurrences : {original_count}")
print(f"Bullets repaired              : {n_bullets}")
print(f"Degree signs repaired         : {n_degrees}")
print(f"Apostrophes repaired          : {n_apostrophes}")
print(f"Residual occurrences          : {text.count(BAD)}")

print()
print(f"Cleaned file  : {OUTPUT_FILE}")
print(f"Residual file : {RESIDUAL_FILE}")
print("=" * 65)

BRAHA — SIEVE A REPAIR
Original mojibake occurrences : 430
Bullets repaired              : 396
Degree signs repaired         : 12
Apostrophes repaired          : 3
Residual occurrences          : 19

Cleaned file  : BR_Curated_18.txt
Residual file : BR_Sieve_A_Residual.txt


In [3]:
import re

INPUT_FILE  = "BR_Curated_18.txt"
OUTPUT_FILE = "BR_Curated_19.txt"

BAD = "Ã¢â‚¬Â¢"

# ------------------------------------------------------------
# READ FILE
# ------------------------------------------------------------

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read()

before = text.count(BAD)

print("=" * 65)
print("BRAHA — SIEVE A FINAL RESIDUAL REPAIR")
print("=" * 65)
print(f"Mojibake occurrences before repair : {before}")
print()


# ------------------------------------------------------------
# EXACT, HUMAN-VERIFIED REPAIRS
# ------------------------------------------------------------

repairs = {
    "anÃ¢â‚¬Â¢ afflicted": "an afflicted",
    "techÃ¢â‚¬Â¢ nically": "technically",
    "has Ã¢â‚¬Â¢been": "has been",
    "ThisÃ¢â‚¬Â¢ is": "This is",
    "the Ã¢â‚¬Â¢most": "the most",
    "been Ã¢â‚¬Â¢described": "been described",

    # Do NOT silently change the separate OCR errors "ts" and "ln"
    # during this mojibake pass.
    "tsÃ¢â‚¬Â¢ln": "ts ln",

    "good Ã¢â‚¬Â¢houses": "good houses",
    "thereÃ¢â‚¬Â¢is": "there is",
    "enablesÃ¢â‚¬Â¢ the": "enables the",
    "a Ã¢â‚¬Â¢strong": "a strong",
    "theÃ¢â‚¬Â¢ significations": "the significations",
    "be Ã¢â‚¬Â¢ the": "be the",
    "good Ã¢â‚¬Â¢deeds": "good deeds",
    "conÃ¢â‚¬Â¢ stantly": "constantly",
    "MarÃ¢â‚¬Â¢ riage": "Marriage",
    "engiÃ¢â‚¬Â¢ neering": "engineering",
}


# ------------------------------------------------------------
# APPLY EXACT REPAIRS
# ------------------------------------------------------------

total_repaired = 0

for wrong, right in repairs.items():
    count = text.count(wrong)

    if count:
        print(f"{count:3d}  {wrong}  -->  {right}")
        text = text.replace(wrong, right)
        total_repaired += count


# ------------------------------------------------------------
# REMOVE STRAY MOJIBAKE AT END OF LINE
#
# Examples found in residual sieve:
#   ... intellectual nature is conferred.  Ã¢â‚¬Â¢
#   ... Happy married life.               Ã¢â‚¬Â¢
# ------------------------------------------------------------

text, trailing_removed = re.subn(
    rf"[ \t]*{re.escape(BAD)}[ \t]*$",
    "",
    text,
    flags=re.MULTILINE
)

if trailing_removed:
    print(f"{trailing_removed:3d}  trailing mojibake marker(s) removed")

total_repaired += trailing_removed


# ------------------------------------------------------------
# WRITE BR_CURATED_19
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(text)


# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

after = text.count(BAD)

print()
print("=" * 65)
print(f"Exact/trailing repairs made       : {total_repaired}")
print(f"Mojibake occurrences before       : {before}")
print(f"Mojibake occurrences remaining    : {after}")
print(f"Output file                       : {OUTPUT_FILE}")
print("=" * 65)

if after == 0:
    print("SIEVE A CLEAN — ZERO MOJIBAKE REMAINS.")
else:
    print("WARNING: MOJIBAKE STILL REMAINS.")
    print("Run Sieve A again on BR_Curated_19.txt.")

BRAHA — SIEVE A FINAL RESIDUAL REPAIR
Mojibake occurrences before repair : 19

  1  anÃ¢â‚¬Â¢ afflicted  -->  an afflicted
  1  techÃ¢â‚¬Â¢ nically  -->  technically
  1  has Ã¢â‚¬Â¢been  -->  has been
  1  ThisÃ¢â‚¬Â¢ is  -->  This is
  1  the Ã¢â‚¬Â¢most  -->  the most
  1  been Ã¢â‚¬Â¢described  -->  been described
  1  tsÃ¢â‚¬Â¢ln  -->  ts ln
  1  good Ã¢â‚¬Â¢houses  -->  good houses
  1  thereÃ¢â‚¬Â¢is  -->  there is
  1  enablesÃ¢â‚¬Â¢ the  -->  enables the
  1  a Ã¢â‚¬Â¢strong  -->  a strong
  1  theÃ¢â‚¬Â¢ significations  -->  the significations
  1  be Ã¢â‚¬Â¢ the  -->  be the
  1  good Ã¢â‚¬Â¢deeds  -->  good deeds
  1  conÃ¢â‚¬Â¢ stantly  -->  constantly
  1  MarÃ¢â‚¬Â¢ riage  -->  Marriage
  1  engiÃ¢â‚¬Â¢ neering  -->  engineering
  2  trailing mojibake marker(s) removed

Exact/trailing repairs made       : 19
Mojibake occurrences before       : 19
Mojibake occurrences remaining    : 0
Output file                       : BR_Curated_19.txt
SIEVE A CLEAN — ZERO MOJIBAKE REMA

In [7]:
import re

INPUT_FILE  = "BR_Curated_20.txt"
OUTPUT_FILE = "BR_Sieve_B_Punctuation2.txt"

# ------------------------------------------------------------
# SIEVE B
# Find suspicious punctuation occurring inside words.
#
# Examples:
#   w! ll
#   del! neated
#   possib! lities
#
# We are particularly interested in !, but also catch
# ? | ~ ^ ` when they occur between/adjacent to letters.
#
# Hyphens and apostrophes are deliberately excluded because
# they are normally legitimate inside English words.
# ------------------------------------------------------------

# punctuation with letters on BOTH sides
pattern_inside = re.compile(
    r"[A-Za-z][!?\|~^`][A-Za-z]"
)

# punctuation followed by whitespace and then letters:
# catches things such as:
#     w! ll
#     del! neated
pattern_split = re.compile(
    r"[A-Za-z][!?\|~^`]\s+[A-Za-z]"
)

hits = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):

        found_inside = pattern_inside.findall(line)
        found_split  = pattern_split.findall(line)

        if found_inside or found_split:
            hits.append(
                (
                    line_no,
                    found_inside,
                    found_split,
                    line.rstrip()
                )
            )


# ------------------------------------------------------------
# WRITE SIEVE OUTPUT
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:

    out.write("BRAHA — SIEVE B: PUNCTUATION INSIDE WORDS\n")
    out.write("=" * 100 + "\n\n")

    for line_no, inside, split, line in hits:

        out.write(f"LINE {line_no}\n")

        if inside:
            out.write(
                "INSIDE WORD : "
                + " | ".join(inside)
                + "\n"
            )

        if split:
            out.write(
                "SPLIT WORD  : "
                + " | ".join(split)
                + "\n"
            )

        out.write(line + "\n")
        out.write("-" * 100 + "\n")


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 65)
print("BRAHA — SIEVE B: PUNCTUATION INSIDE WORDS")
print("=" * 65)
print(f"Lines flagged : {len(hits)}")
print(f"Output file   : {OUTPUT_FILE}")
print("=" * 65)

BRAHA — SIEVE B: PUNCTUATION INSIDE WORDS
Lines flagged : 6
Output file   : BR_Sieve_B_Punctuation2.txt


In [6]:
INPUT_FILE  = "BR_Curated_19.txt"
OUTPUT_FILE = "BR_Curated_20.txt"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read()

repairs = {
    "w! ll": "will",
    "personality! n": "personality in",
    "horoscope! n": "horoscope in",
    "del! neated": "delineated",
    "blbl! ography": "blbliography",   # preserve other OCR error for later
    "extraord! nartly": "extraordinarily",
    "noted! n": "noted in",
    "s! gnlflcant": "significant",
    "l! ght": "light",
    "s! m! larly": "similarly",
    "the! st house": "the 1st house",
    "Mr! tyu": "Mrityu",
    "rel! gious": "religious",
    "L! terary": "Literary",
    "s! bl! ng": "sibling",
    "rel! g! ous": "religious",
    "alimony! f": "alimony if",
    "phllosoph! cal": "phllosophical", # leave phll... for later
    "l! ke": "like",
    "br! lliant": "brilliant",
    "Kr! shnamurtl": "Krishnamurtl",   # leave final l for later
    "moneylend! ng": "moneylending",
    "intell! gent": "intelligent",
    "holyp! lgrlmages": "holypilgrlmages", # other OCR remains
    "amb! tlons": "ambitlons",         # other OCR remains
    "muc! fof": "much fof",            # only fix ! -> h/i issue conservatively
    "social! ife": "social life",
    "l! ttle": "little",
    "ab! lity": "ability",
    "uncann! ly": "uncannily",
    "aspects! ts": "aspects its",
    "l! beration": "liberation",
    "ph! losophy": "philosophy",
    "wh! le": "while",
    "is! nArles": "is in Arles",
    "curta! led": "curtailed",
}

total = 0

for wrong, right in repairs.items():
    count = text.count(wrong)
    if count:
        print(f"{count:3d}  {wrong}  -->  {right}")
        text = text.replace(wrong, right)
        total += count

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(text)

print()
print("=" * 65)
print("BRAHA — SIEVE B REPAIR")
print("=" * 65)
print(f"Replacements made : {total}")
print(f"Output file       : {OUTPUT_FILE}")
print("=" * 65)

 21  w! ll  -->  will
  1  personality! n  -->  personality in
  1  horoscope! n  -->  horoscope in
  1  del! neated  -->  delineated
  1  blbl! ography  -->  blbliography
  1  extraord! nartly  -->  extraordinarily
  1  noted! n  -->  noted in
  1  s! gnlflcant  -->  significant
  1  l! ght  -->  light
  1  s! m! larly  -->  similarly
  1  the! st house  -->  the 1st house
  1  Mr! tyu  -->  Mrityu
  1  rel! gious  -->  religious
  1  L! terary  -->  Literary
  1  s! bl! ng  -->  sibling
  1  rel! g! ous  -->  religious
  1  alimony! f  -->  alimony if
  1  phllosoph! cal  -->  phllosophical
  1  l! ke  -->  like
  1  br! lliant  -->  brilliant
  1  Kr! shnamurtl  -->  Krishnamurtl
  1  moneylend! ng  -->  moneylending
  1  intell! gent  -->  intelligent
  1  holyp! lgrlmages  -->  holypilgrlmages
  1  amb! tlons  -->  ambitlons
  1  muc! fof  -->  much fof
  1  social! ife  -->  social life
  1  l! ttle  -->  little
  1  ab! lity  -->  ability
  1  uncann! ly  -->  uncannily
  1  asp

In [8]:
import re

INPUT_FILE  = "BR_Curated_20.txt"
OUTPUT_FILE = "BR_Sieve_C_DigitsInWords.txt"

# ------------------------------------------------------------
# SIEVE C
# Find digits embedded inside alphabetic words.
#
# Examples:
#   directiona1
#   financia1
#   l0rd
#   5ign
#
# Discovery only. NO automatic correction.
# ------------------------------------------------------------

# Case 1:
# letter(s) followed by digit(s), possibly followed by letters
#
# Examples:
#   directiona1
#   l0rd
#   planet5
#
pattern_letter_digit = re.compile(
    r"\b[A-Za-z]+[0-9]+[A-Za-z]*\b"
)

# Case 2:
# digit(s) followed immediately by letters
#
# Examples:
#   5ign
#   1ord
#
# But avoid normal constructions such as:
#   10th
#   11th
#   12th
#   18-year
#
pattern_digit_letter = re.compile(
    r"\b[0-9]+[A-Za-z]+\b"
)

# Normal numeric words which we do NOT want flagged
normal_numeric_tokens = re.compile(
    r"^[0-9]+(?:st|nd|rd|th)$",
    re.IGNORECASE
)

hits = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:

    for line_no, line in enumerate(f, start=1):

        suspects = []

        # Letter -> digit corruption
        for match in pattern_letter_digit.finditer(line):
            suspects.append(match.group())

        # Digit -> letter corruption
        for match in pattern_digit_letter.finditer(line):

            token = match.group()

            # Ignore legitimate ordinals:
            # 1st, 2nd, 3rd, 4th, 10th, 11th, 12th etc.
            if normal_numeric_tokens.match(token):
                continue

            suspects.append(token)

        if suspects:
            hits.append(
                (
                    line_no,
                    sorted(set(suspects)),
                    line.rstrip()
                )
            )


# ------------------------------------------------------------
# WRITE OUTPUT
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:

    out.write("BRAHA — SIEVE C: DIGITS EMBEDDED IN WORDS\n")
    out.write("=" * 100 + "\n\n")

    for line_no, suspects, line in hits:

        out.write(f"LINE {line_no}\n")
        out.write(
            "SUSPECT : "
            + " | ".join(suspects)
            + "\n"
        )

        out.write(line + "\n")
        out.write("-" * 100 + "\n")


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 65)
print("BRAHA — SIEVE C: DIGITS EMBEDDED IN WORDS")
print("=" * 65)
print(f"Lines flagged : {len(hits)}")
print(f"Output file   : {OUTPUT_FILE}")
print("=" * 65)

BRAHA — SIEVE C: DIGITS EMBEDDED IN WORDS
Lines flagged : 66
Output file   : BR_Sieve_C_DigitsInWords.txt


In [9]:
INPUT_FILE  = "BR_Curated_20.txt"
OUTPUT_FILE = "BR_Curated_21.txt"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read()

repairs = {
    # --------------------------------------------------------
    # 1 misread as lowercase l
    # --------------------------------------------------------
    "directiona1": "directional",
    "Professiona1status": "Professional status",
    "Physica1": "Physical",
    "mythologica1": "mythological",
    "menta1": "mental",
    "Menta1": "Mental",
    "Sexua1": "Sexual",
    "sexua1": "sexual",
    "wil1": "will",
    "numerica1": "numerical",
    "natura1": "natural",
    "empirica1": "empirical",
    "factua1": "factual",
    "physica1": "physical",
    "uphil1": "uphill",
    "spiritua1": "spiritual",
    "educationa1status": "educational status",
    "educationa1": "educational",
    "metaphysica1": "metaphysical",
    "ordirectiona1": "or directional",
    "financia1": "financial",
    "peacefu1": "peaceful",
    "perpetua1": "perpetual",

    # --------------------------------------------------------
    # Other obvious digit-for-letter OCR
    # --------------------------------------------------------
    "t9 be": "to be",
    "1t is considered": "it is considered",
    "8agittarlus": "Sagittarlus",   # fix digit only; leave other OCR
    "giv3s": "gives",

    # --------------------------------------------------------
    # Obvious spacing / corruption exposed by this sieve
    # --------------------------------------------------------
    "house1>": "houses",
    "the6th house": "the 6th house",
    "the12th house": "the 12th house",
    "of20-29": "of 20-29",
    "If2 marakas": "If 2 marakas",

    # Venu13 is clearly Venus in this sentence
    "Venu13 In a sign of Mars": "Venus In a sign of Mars",
}

total = 0

print("=" * 70)
print("BRAHA — SIEVE C REPAIR")
print("=" * 70)

for wrong, right in repairs.items():

    count = text.count(wrong)

    if count:
        print(f"{count:3d}  {wrong}  -->  {right}")
        text = text.replace(wrong, right)
        total += count


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(text)

print()
print("=" * 70)
print(f"Replacements made : {total}")
print(f"Output file       : {OUTPUT_FILE}")
print("=" * 70)

BRAHA — SIEVE C REPAIR
 14  directiona1  -->  directional
  1  Professiona1status  -->  Professional status
  1  Physica1  -->  Physical
  1  mythologica1  -->  mythological
  4  menta1  -->  mental
  1  Menta1  -->  Mental
  1  Sexua1  -->  Sexual
  9  sexua1  -->  sexual
  2  wil1  -->  will
  1  numerica1  -->  numerical
  1  natura1  -->  natural
  1  empirica1  -->  empirical
  1  factua1  -->  factual
  3  physica1  -->  physical
  3  uphil1  -->  uphill
  2  spiritua1  -->  spiritual
  1  educationa1status  -->  educational status
  1  educationa1  -->  educational
  1  financia1  -->  financial
  1  peacefu1  -->  peaceful
  1  perpetua1  -->  perpetual
  1  t9 be  -->  to be
  1  1t is considered  -->  it is considered
  1  8agittarlus  -->  Sagittarlus
  1  giv3s  -->  gives
  1  house1>  -->  houses
  1  the6th house  -->  the 6th house
  1  the12th house  -->  the 12th house
  1  of20-29  -->  of 20-29
  1  If2 marakas  -->  If 2 marakas
  1  Venu13 In a sign of Mars  -->  